# D2 · Components, ports, topology and margins

The powertrain is described as **components joined through typed ports**, with counts ("2 lanes per rotor, 2 buses,
2 battery strings"). This tutorial shows what that description is, what it does for you, and, honestly, what it does
*not* do inside the Halo sizing.

**What you will learn**
1. The component contract: `get_mass()`, `get_limits()`, `evaluate(...)`, and nothing else.
2. Ports, `Topology.add/connect`, buses, multiplicity and combiners; the wiring errors it rejects.
3. `connection_residuals`: the equations a topology implies.
4. How the Halo flight point actually satisfies those equations (by construction), and why applying the residuals
   in a failure state would be wrong.
5. Normalized, named margins: how every limit becomes a constraint with a readable name.

**Run time:** about a minute (it solves single flight points on the sized aircraft).

In [1]:
import sys
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(repo_root / "tutorials"))
from tutorial_helpers import *          # paths, plot style, check(), lb(), baseline(), capture_opti()

from dataclasses import replace
from examples.halo_sizing import HaloRequirements, build_halo_aircraft, build_halo_aerodynamics

ref, assumptions, requirements = baseline(), baseline_assumptions(), HaloRequirements()
# The sized v3.6 aircraft, rebuilt with numbers (the solved design) instead of optimization variables.
aircraft = build_halo_aircraft(ref.design, requirements, assumptions)
topology = aircraft.powertrain.topology

## 1. The component contract

Every powertrain component is a frozen dataclass with three methods (`docs/CODING_CONVENTIONS.md`):

| Method | Returns |
|---|---|
| `get_mass()` | its mass (an expression if its ratings are variables) |
| `get_limits()` | a small dataclass of limits (max torque, max speed, voltage window, ...) |
| `evaluate(...)` | its operating state for given physical inputs (speed, torque, voltage, current, ...) |

A component **never** creates an optimization variable, never clips a value, and never runs its own solver. Take the
sized motor lane:

In [2]:
motor = topology.instances["motor"].component
print(type(motor).__name__, "| loss model:", type(motor.loss_model).__name__, "| mass model:", type(motor.mass_model).__name__)
print(f"mass {motor.get_mass():.1f} kg, limits: {motor.get_limits()}")
state = motor.evaluate(150.0, 1200.0, 700.0)          # speed rad/s, torque N m, bus voltage V
print(state)
check("electrical power = shaft power + loss", abs(state.power_electric_W - (150 * 1200 + state.power_loss_W)) < 1e-6)

Motor | loss model: McDonaldMotorLossModel | mass model: UnitMachineMassModel
mass 106.4 kg, limits: MotorLimits(power_rated_W=527787.5658030852, max_speed_rad_s=261.79938779914943, max_torque_Nm=3000.0, min_voltage_V=400.0, max_voltage_V=900.0)
MotorResult(power_shaft_W=180000.0, power_electric_W=187790.97756813848, current_A=268.2728250973407, power_loss_W=7790.977568138485)
PASS  electrical power = shaft power + loss


Because `evaluate` is plain arithmetic in `aerosandbox.numpy`, the same call inside the sizing receives Opti
variables (the motor's peak-efficiency torque and speed are design variables; its operating speed and torque are
per-point quantities) and returns expressions. The *caller* decides what is a variable and what is constrained.

## 2. Ports and topology

Each component type declares typed ports in `powertrain/ports.py` (kept outside the classes so the component
interface stays clean): a **shaft** port carries speed and torque, an **electrical** port voltage and current, a
**fuel** port fuel flow; each port has a nominal direction (IN or OUT). A `Topology` holds named instances with a
`count`, electrical buses, and connections. It is a *description*: no variables, no constraints, no solving.

In [3]:
from aircraft_closure.core.topology import Topology
from aircraft_closure.core.ports import Domain
from aircraft_closure.powertrain.ports import port_specs_for

for name in ("turboshaft", "generator", "battery", "motor", "gearbox", "propulsor"):
    component = topology.instances[name].component
    print(f"{type(component).__name__:<24}", [f"{p.name}:{p.domain.value}:{p.direction.value}" for p in port_specs_for(component)])

SimpleTurboshaft         ['fuel:fuel:in', 'shaft:mechanical:out']
Generator                ['shaft:mechanical:in', 'electrical:electrical:out']
EquivalentCircuitBattery ['electrical:electrical:out']
Motor                    ['electrical:electrical:in', 'shaft:mechanical:out']
Gearbox                  ['shaft_in:mechanical:in', 'shaft_out:mechanical:out']
MomentumProfileRotor     ['shaft:mechanical:in']


The topology validates every connection **immediately**, before any solve. Three wiring mistakes it rejects:

In [4]:
c = {name: topology.instances[name].component for name in ("turboshaft", "generator", "motor", "gearbox", "propulsor")}

def try_wiring(label, build):
    t = Topology()
    for name, component in c.items():
        t.add(name, component, port_specs_for(component), count=4 if name == "motor" else 2)
    t.add_bus("bus")
    try:
        build(t)
        print(f"accepted  {label}")
    except ValueError as error:
        print(f"REJECTED  {label}\n          -> {error}")

try_wiring("shaft port to an electrical bus", lambda t: t.connect("turboshaft.shaft", "bus"))
try_wiring("two OUT ports joined", lambda t: t.connect("turboshaft.shaft", "motor.shaft"))
try_wiring("4 motors on 2 gearboxes without combine", lambda t: t.connect("motor.shaft", "gearbox.shaft_in"))
try_wiring("4 motors on 2 gearboxes as a combiner", lambda t: t.connect("motor.shaft", "gearbox.shaft_in", combine=True))

REJECTED  shaft port to an electrical bus
          -> Cannot connect mechanical port 'turboshaft.shaft' to electrical bus 'bus'.
REJECTED  two OUT ports joined
          -> Direct connections join one OUT and one IN port: 'turboshaft.shaft' and 'motor.shaft' are both out.
REJECTED  4 motors on 2 gearboxes without combine
          -> Direct connection between counts 4 and 2 needs combine=True (a combiner or splitter) or a bus.
accepted  4 motors on 2 gearboxes as a combiner


**Multiplicity.** An instance with `count = n` is *n identical copies sharing one set of port values* (symmetric
operation). A direct connection between different counts must be declared a **combiner or splitter**
(`combine=True`): efforts (speed, voltage) are equal and the total flow is conserved,
`count_source × flow_source = count_target × flow_target`. Two lane motors on one rotor's gearbox is a combiner; one
battery split into isolated strings is a splitter. A **bus** is a junction for any number of electrical ports: equal
voltages and Kirchhoff's current law, weighted by counts. A bus can itself have a count (cross-strapped buses lumped
as one junction).

Here is the sized Halo network, built by `build_series_hybrid(...)` from the components and a `RedundancyLayer`:

In [5]:
print("instances (count):")
for name, instance in topology.instances.items():
    print(f"   {name:<20}{type(instance.component).__name__:<26}x{instance.count}")
print("buses:", {name: bus.count for name, bus in topology.buses.items()})
print("connections:")
for connection in topology.connections:
    print(f"   {connection.source:<28} -> {connection.target:<22}{'(combine)' if connection.combine else ''}")
check("2 lanes per rotor: 4 lane motors on 2 rotors", topology.instances["motor"].count == 4)

instances (count):
   turboshaft          SimpleTurboshaft          x2
   generator           Generator                 x2
   battery             EquivalentCircuitBattery  x1
   motor               Motor                     x4
   gearbox             Gearbox                   x2
   propulsor           MomentumProfileRotor      x2
   generator_gearbox   Gearbox                   x2
   protection_string   ProtectionUnit            x2
   protection_bus_tie  ProtectionUnit            x1
   cable_bus_tie       Cable                     x1
buses: {'bus': 2}
connections:
   turboshaft.shaft             -> generator_gearbox.shaft_in
   generator_gearbox.shaft_out  -> generator.shaft       
   battery.electrical           -> protection_string.input(combine)
   protection_bus_tie.output    -> cable_bus_tie.input   
   generator.electrical         -> bus                   
   protection_string.output     -> bus                   
   motor.electrical             -> bus                   
   motor.s

Two details worth pointing out:

- The **bus tie** (`protection_bus_tie` → `cable_bus_tie`) is connected only to itself: its outer ports are
  *boundaries*. The tie is normally open, so in normal operation it carries no current. A bus failure is not a
  rewiring: it is a flight condition (section 4).
- The **turboshaft fuel port** is unconnected (a boundary): fuel flow is read from the engine and integrated by the
  mission.

The topology also feeds the mass: `PowertrainInstallation` books `count × get_mass()` for each instance at its
installed location, so "2 lanes per rotor" automatically puts four motors' mass into the weight and CG.

In [6]:
masses = dict(ref.powertrain_masses_kg)
check("installed motor mass = count x unit mass",
      abs(masses["motor"] - topology.instances["motor"].count * motor.get_mass()) < 1e-6)

PASS  installed motor mass = count x unit mass


## 3. The equations a topology implies

`connection_residuals(topology, port_values)` turns the description into labelled equations: for each direct
connection, source and target fields equal (or, for a combiner, efforts equal and total flow conserved); for each bus,
equal voltages and a net current of zero. The caller supplies a value for every connected port and constrains the
residuals to zero. `examples/series_hybrid_point.py` does exactly that, and reproduces the hand-written coupled point
of `series_hybrid_point_explicit.py` to 2e-10.

In [7]:
from aircraft_closure.core.ports import ElectricalPortValue, MechanicalPortValue
from aircraft_closure.core.topology import connection_residuals

t = Topology()
for name in ("motor", "gearbox"):
    t.add(name, c[name], port_specs_for(c[name]), count=4 if name == "motor" else 2)
t.connect("motor.shaft", "gearbox.shaft_in", combine=True)
values = {"motor.shaft": MechanicalPortValue(speed_rad_s=150.0, torque_Nm=1000.0),
          "gearbox.shaft_in": MechanicalPortValue(speed_rad_s=150.0, torque_Nm=2000.0)}
for residual in connection_residuals(t, values):
    print(f"{residual.label:<45} = {residual.value}")

motor.shaft->gearbox.shaft_in speed_rad_s     = 0.0
motor.shaft->gearbox.shaft_in torque_Nm (total) = 0.0


Two lanes at 1,000 N·m each feed one gearbox input of 2,000 N·m: equal speed, and 4 × 1,000 − 2 × 2,000 = 0.

## 4. How the Halo sizing satisfies the connections: by construction

Here is a point most readers miss. **The Halo flight point never applies `connection_residuals`.** `build_flight_point`
(`src/aircraft_closure/performance/flight_point.py`) writes the connections *by construction*:

- one `voltage_bus_V` expression is used for every port on the bus;
- shaft speeds follow the gear ratio (`speed_motor = ratio × speed_rotor`), torques pass through the gearbox model;
- the gearbox input torque is split over the active lanes;
- the bus current balance is replaced by two **power** balances: battery bus power = h × demand, and generator bus
  power × active generators = (1 − h) × demand, where h is the battery share.

It still builds the full `port_values` dictionary, but only to evaluate the **operating margins** (section 5). Writing
connections by construction removes variables and equalities, so the problem is smaller and better conditioned.

Check it: solve the 4,000 ft hover requirement on the sized aircraft (exactly as the sizing builds it: a 60 s hover
from coolant temperature, at the battery share the sizing chose), capture the port values, and evaluate the
residuals the topology implies.

In [8]:
from aircraft_closure.performance import flight_point as fp
from aircraft_closure.thermal.heat import coolant_temperatures_C

aerodynamics = build_halo_aerodynamics(requirements, assumptions)
captured = {}
_operating_margins = fp.operating_margins

def spy(topology_, port_values, *args, **kwargs):          # record what the point passes to the margins
    captured["port_values"] = port_values
    return _operating_margins(topology_, port_values, *args, **kwargs)

def solve_hover(**degraded):
    fp.operating_margins = spy
    try:
        opti = asb.Opti()
        condition = fp.FlightCondition(mode="hover", altitude_m=requirements.altitude_hover_m,
                                       thrust_to_weight=requirements.thrust_to_weight_hover, soc=0.8,
                                       label="hover", **degraded)
        point = fp.build_flight_point(opti, aircraft, aerodynamics, condition, ref.mass_takeoff_kg,
                                      hybridization_electric=ref.hovers[0].hybridization, duration_s=60.0,
                                      temperature_start_C=coolant_temperatures_C(aircraft.powertrain))
        opti.minimize(point.fuel_flow_kg_s)          # the point is square: this only picks the solver's path
        solution = opti.solve(verbose=False)
    finally:
        fp.operating_margins = _operating_margins
    residuals = {res.label: float(solution.value(res.value))
                 for res in connection_residuals(topology, captured["port_values"])}
    return point, solution, residuals

point, sol, residuals = solve_hover()
voltage_bus_V = sol.value(captured["port_values"]["motor.electrical"].voltage_V)
power_fan_W = sol.value(point.thermal.cooling.power_fan_W)
for label, value in sorted(residuals.items(), key=lambda item: -abs(item[1]))[:5]:
    print(f"{label:<55}{value:12.4g}")
print(f"\nbus current residual x bus voltage = {residuals['bus current_A'] * voltage_bus_V:.2f} W; "
      f"heat-exchanger fan power = {power_fan_W:.2f} W")
check("every shaft residual is zero by construction",
      max(abs(v) for k, v in residuals.items() if "bus" not in k) < 1e-6)
check("the only non-zero residual is the cooler fan's current (the fan draws from the bus but has no port)",
      abs(residuals["bus current_A"] * voltage_bus_V - power_fan_W) < 1e-3)

bus current_A                                               0.07132
turboshaft.shaft->generator_gearbox.shaft_in speed_rad_s           0
turboshaft.shaft->generator_gearbox.shaft_in torque_Nm            0
generator_gearbox.shaft_out->generator.shaft speed_rad_s           0
generator_gearbox.shaft_out->generator.shaft torque_Nm            0

bus current residual x bus voltage = 52.73 W; heat-exchanger fan power = 52.73 W
PASS  every shaft residual is zero by construction
PASS  the only non-zero residual is the cooler fan's current (the fan draws from the bus but has no port)


Every shaft and voltage connection holds to round-off. The single non-zero residual is the bus current, and it is
exactly the heat-exchanger fan: in hover the fan's electrical power joins the bus demand in the power balance, but the
fan is not a topology instance with a port, so a port-level Kirchhoff sum does not see it. Nothing is wrong
physically; it shows that the power balance, not the residual list, is the source of truth in the sizing.

Now the important case. Fail one bus (its lanes are lost; its sources feed the other bus through the tie). This is
only to compare residuals, so it reuses the 4,000 ft hover conditions; the sizing's own bus-out case is a sea-level
hover starting from the 30 % reserve (core tutorial 19, failure hovers).

In [9]:
point_out, sol_out, residuals_out = solve_hover(count_buses_failed=1)
for label, value in sorted(residuals_out.items(), key=lambda item: -abs(item[1]))[:3]:
    print(f"{label:<55}{value:12.4g}")
torque_lane = sol_out.value(point_out.torque_motor_Nm)
print(f"\none active lane per rotor carries {torque_lane:,.0f} N m, but the topology says 2 lanes per rotor:")
print(f"  residual = 4 lanes x {torque_lane:,.0f} - 2 gearboxes x {torque_lane:,.0f} = {2 * torque_lane:,.0f} N m")
check("in the degraded state the topology's residuals are NOT zero", abs(residuals_out["motor.shaft->gearbox.shaft_in torque_Nm (total)"]) > 1)

motor.shaft->gearbox.shaft_in torque_Nm (total)                6787
bus current_A                                                 -2156
turboshaft.shaft->generator_gearbox.shaft_in speed_rad_s           0

one active lane per rotor carries 3,393 N m, but the topology says 2 lanes per rotor:
  residual = 4 lanes x 3,393 - 2 gearboxes x 3,393 = 6,787 N m
PASS  in the degraded state the topology's residuals are NOT zero


The point is physically consistent (one lane carries the gearbox input torque), but `connection_residuals` uses the
**installed** counts, not the **active** ones. So in a failure state the residuals would be wrong, not just
redundant. That is the design: the topology describes the installed network; a failure (`active_lane_count`,
`count_buses_failed`, `active_battery_string_count`, `active_generator_count`) is a property of the *flight
condition*, applied by the flight point (`powertrain/redundancy.py: degraded_state`).

So, inside the Halo sizing, the topology is used for:

1. **validation** at build time (domains, directions, counts);
2. **multiplicity**: installed counts, and the base for active counts in failure states;
3. **lookup** of components by instance name;
4. **mass and CG**: count × mass at each installed location;
5. **operating margins** at every point (next section);
6. **naming** of heat loads and margins.

## 5. Margins: every limit, normalized and named

`core/margins.py` is 40 lines and used everywhere:

```python
margin_below(label, value, limit) = (limit - value) / limit     # value <= limit
margin_above(label, value, limit) = (value - limit) / limit     # value >= limit
```

A margin ≥ 0 is satisfied; 0.1 means 10 % headroom. Margins *report*; they never clip or resize. The sizing
constrains `margin.value >= 0` for all of them, and after the solve reports the ones at zero as binding. Normalizing
by the limit gives IPOPT order-one constraints and gives a human a direct reading.

`operating_margins(topology, port_values, atmosphere)` produces the per-component limits at a point:

In [10]:
margins = sorted(((m.label, float(sol.value(m.value))) for m in point.margins), key=lambda item: item[1])
print(f"{len(margins)} margins at the 4,000 ft hover; the 10 tightest:")
for label, value in margins[:10]:
    print(f"   {value:+.4f}  {label}")
check("the rotor and its gearbox are at their power limits in this hover (they size the drive)",
      abs(dict(margins)["hover: propulsor power_shaft_W"]) < 1e-6)

21 margins at the 4,000 ft hover; the 10 tightest:
   -0.0000  hover: propulsor power_shaft_W
   -0.0000  hover: gearbox power_input_W
   +0.1473  hover: motor speed_rad_s
   +0.1615  hover: battery max_voltage_V
   +0.1785  hover: generator max_voltage_V
   +0.1785  hover: motor max_voltage_V
   +0.2973  hover: turboshaft power_shaft_W
   +0.3069  hover: battery discharge_current_A
   +0.3612  hover: generator_gearbox power_input_W
   +0.3612  hover: generator torque_Nm
PASS  the rotor and its gearbox are at their power limits in this hover (they size the drive)


| Component | Operating margins (`powertrain/compatibility.py`) |
|---|---|
| Motor, generator | torque ≤ max; speed ≤ max; bus voltage inside the machine's window; shaft power ≤ rating *unless* a thermal model is on (then temperature limits instead) |
| Battery (equivalent circuit) | discharge and charge current ≤ ratings; terminal voltage inside N_s × [2.5, 4.2] V |
| Turboshaft | shaft power ≤ power **available at this point's altitude and temperature** |
| Gearbox, rotor | input / shaft power ≤ rating |
| Cable, protection | current² ≤ rating² (smooth for both signs); cable partial-discharge voltage at altitude |
| Inverter, DC/DC (electrical layer) | power ≤ rating; DC voltage window and derated blocking voltage |

The labels are prefixed with the flight condition, so the binding list of the whole sizing reads like
`engine-out hover 3/3: battery end voltage_V`.

## Why it is built this way

- **Describe the network as data.** "Two lanes per rotor, two buses, two strings" becomes a declaration rather than
  hand-written bookkeeping: counts flow into mass, margins, heat loads and failure states from one place, and wiring
  errors are caught before a solve (`docs/decisions/002`, `032`).
- **The graph never owns optimization.** It is deliberately lighter than an OpenMDAO `Group`: no variables, no
  solver, no execution order (`docs/ARCHITECTURE.md`). CasADi resolves the coupled equations.
- **Connections by construction in the production point.** Fewer variables and equalities, better conditioned; the
  residual route is kept (and tested) and reproduces the same answer on the reference example.
- **Failures are conditions, not topologies.** Redundancy is a topology choice; a failure reuses the same network
  with fewer active units, so a failure case is one more flight point in the same problem.
- **Normalized margins** for solver conditioning, comparability across physics, and readable binding constraints.

## What it can't do

- **Symmetric multiplicity only.** `count = n` means n identical copies with identical states. One rotor degraded
  while the other is healthy (and the roll trim that implies) cannot be represented; failures are applied to every
  rotor at once, which is conservative for power and heat (`NEXT_STEPS.md` §6).
- **The residuals are not the production equations.** As shown, they ignore the cooler fan and use installed counts.
  If you add a component to the network, you must also add its physics to `build_flight_point`; the topology alone
  will not couple it.
- **Adjacent-rating design checks are not enforced in the Halo sizing.** `compatibility.design_margins` (does the
  gearbox tolerate the motor's maximum torque? is the bus voltage window inside every machine's window?) exists and
  is tested, but the sizing does not constrain it; it relies on operating margins at the points it flies.
- **No torque-speed envelope** for machines: maximum torque, maximum speed and rating are independent bounds.

## What's next

- Asymmetric failures and double failures (a lane and an engine), and an interconnect shaft against electrical
  cross-strapping as an architecture study (`README`, propulsion work package).
- A design-margin pass (the adjacent-rating checks) reported after each sizing, or constrained where it matters.

## Check yourself

<details><summary>1. What does `count=4` on the motor instance mean physically and mathematically?</summary>

Four identical lane motors (two per rotor) that share one set of port values: same speed, same torque, same
current. Mass and current into the bus scale by 4. It cannot represent one motor behaving differently.
</details>

<details><summary>2. If the topology's residuals are not applied, what makes the network consistent in the sizing?</summary>

`build_flight_point` writes each connection by construction (shared bus voltage, gear kinematics, torque passed
through) and closes the bus with two power-balance equalities (battery share h and generator share 1 − h of the
demand, including fan power and tie losses).
</details>

<details><summary>3. Why would applying `connection_residuals` in a bus-out hover be wrong?</summary>

The residuals use installed counts (2 lanes per rotor), while the degraded point has 1 active lane per rotor; the
combiner residual becomes 4 × T − 2 × T ≠ 0 even though the physics is right. Failure states live in the flight
condition, not the topology.
</details>

In [11]:
check_summary()

7 of 7 checks passed
